# Visualizacao e tratamento de dados


In [134]:
import pandas as pd
import sqlite3

In [135]:
# Importado o primeiro dataframe
df_mining = pd.read_csv("./data/enderecos.csv")
df_mining.head()

,nome,lat,lon,tipo,capidade,rua,cep,road,suburb,city,state,postcode
0,NaN,40.447447,-3.627856,parking,NaN,Calle Doctor Zamenhof,28037,Calle Doctor Zamenhof,San Blas - Canillejas,Madrid,Comunidad de Madrid,28037
1,NaN,-13.005580,-38.523792,parking,NaN,Avenida Centenário,40140-400,Avenida Centenário,Chame-Chame,Salvador,Bahia,40140-400
2,NaN,-13.005999,-38.523991,parking,NaN,Avenida Centenário,40150-080,Avenida Centenário,Chame-Chame,Salvador,Bahia,40150-080
3,NaN,-13.006483,-38.525508,parking,NaN,Rua Cardeal Enrico Dante,40140-333,Rua Cardeal Enrico Dante,Barra,Salvador,Bahia,40140-333
4,NaN,-13.003179,-38.506724,parking,NaN,Avenida Milton Santos,40170-110,Avenida Milton Santos,Ondina,Salvador,Bahia,40170-110


In [136]:
# Vi que de cara a primeira linha tem Madri, entao preferi comecar com valores unicos
df_mining["city"].unique()

<StringArray>
['Madrid', 'Salvador', 'Beja', 'Serpa', 'Torres Novas']
Length: 5, dtype: str

In [137]:
stranger_cities = ['Beja', 'Serpa', 'Torres Novas', 'Madrid']
df_mining = df_mining[~df_mining['city'].isin(stranger_cities)]
df_mining["city"].unique()

<StringArray>
['Salvador']
Length: 1, dtype: str

In [138]:
df_mining.drop(columns=["nome", "road", "state", "postcode", "cep", "suburb"], inplace=True)

In [139]:
df_mining.head(5)

,lat,lon,tipo,capidade,rua,city
1,-13.005580,-38.523792,parking,NaN,Avenida Centenário,Salvador
2,-13.005999,-38.523991,parking,NaN,Avenida Centenário,Salvador
3,-13.006483,-38.525508,parking,NaN,Rua Cardeal Enrico Dante,Salvador
4,-13.003179,-38.506724,parking,NaN,Avenida Milton Santos,Salvador
5,-12.996014,-38.464123,parking,NaN,Rua Alberto Silva,Salvador


In [140]:
# Preparando o ambiente python para receber dados SQL
conn_sqlite = sqlite3.connect('./data/estacionamentos.db')
cursor = conn_sqlite.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tabelas = cursor.fetchall()

tabelas

[('sqlite_sequence',), ('locais',)]

In [141]:
# Importando os dados do "ondeestacionar.com.br"
cursor.execute("SELECT * FROM locais")
dados = cursor.fetchall()

df_main = pd.read_sql_query("SELECT * FROM locais", conn_sqlite)

In [142]:
df_main.head()

,id,nome,lat,lng,tipo
0,1,Pituba Parque Center (Bahia Park),-12.995769,-38.467289,Privado
1,2,Rua João das Botas (Bahia Park),-12.991060,-38.520452,Privado
2,3,Cei Clínica Medica (Bahia Park),-12.970220,-38.500550,Privado
3,4,Hospital Aristides Maltez (HAM) (Bahia Park),-12.990781,-38.485948,Privado
4,5,Bahia Marina (Bahia Park),-12.979939,-38.519480,Privado


In [143]:
df_main.columns

Index(['id', 'nome', 'lat', 'lng', 'tipo'], dtype='str')

In [144]:
# Padronizando colunas
df_main = df_main.rename(columns={'lng': 'lon'})

In [145]:
# Arredondando para melhor precisao
df_mining['lat'] = df_mining['lat'].round(4)
df_mining['lon'] = df_mining['lon'].round(4)

df_main['lat'] = df_main['lat'].round(4)
df_main['lon'] = df_main['lon'].round(4)

In [146]:
df_mining.columns

Index(['lat', 'lon', 'tipo', 'capidade', 'rua', 'city'], dtype='str')

In [147]:
# Usando Merge para buscar se existem as mesmas coordenadas no banco
comparacao = df_mining.merge(
    df_main[['lat', 'lon']],
    on=['lat', 'lon'],
    how='left',
    indicator=True
)
est_faltantes = comparacao[comparacao['_merge'] == 'left_only']


In [148]:
# Resultado
est_faltantes = pd.DataFrame(est_faltantes)
est_faltantes

,lat,lon,tipo,capidade,rua,city,_merge
0,-13.0056,-38.5238,parking,NaN,Avenida Centenário,Salvador,left_only
1,-13.0060,-38.5240,parking,NaN,Avenida Centenário,Salvador,left_only
2,-13.0065,-38.5255,parking,NaN,Rua Cardeal Enrico Dante,Salvador,left_only
3,-13.0032,-38.5067,parking,NaN,Avenida Milton Santos,Salvador,left_only
4,-12.9960,-38.4641,parking,NaN,Rua Alberto Silva,Salvador,left_only
...,...,...,...,...,...,...,...
78,-12.9374,-38.4127,parking,NaN,Rua Adalgisa Souza Pinto,Salvador,left_only
79,-12.9981,-38.4682,parking,NaN,Avenida Antônio Carlos Magalhães,Salvador,left_only
80,-12.9406,-38.4068,parking,NaN,Rua Le Parc,Salvador,left_only
81,-12.9977,-38.4884,parking,NaN,Rua Waldemar Falcão,Salvador,left_only


In [149]:
# ToDo!!!
# Minerar capacidade dos estacionamentos
# Minerar o tipo de estacionamento Publico/Privado
# Falar com Luiza sobre o que eu posso incrementar